# Week 94, Day 4: Building CPython from source

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w094d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('runs.py', 'w', encoding='utf-8') as f:
    f.write('BASELINE = """== Tests result: FAILURE ==\n\n433 tests OK.\n\n1 test failed:\n    test_ssl\n\n5 tests skipped:\n    test_devpoll test_kqueue test_msvcrt\n    test_winreg test_winsound\n\nTotal duration: 4 min 12 sec\n"""\n\nAFTER_CHANGE = """== Tests result: FAILURE ==\n\n431 tests OK.\n\n3 tests failed:\n    test_ssl test_textwrap\n    test_pydoc\n\n4 tests skipped:\n    test_devpoll test_kqueue test_msvcrt\n    test_winreg\n\nTotal duration: 4 min 31 sec\n"""\n')

**Predict before you run:** what will this print?



In [ ]:
import sys

print(sys.version_info[:2] >= (3, 12), sys.implementation.name)
print(type(sys.version), "(" in sys.version)

> **Key idea.** Build with `./configure --with-pydebug && make -j8` (Linux, then `./python`; macOS, then `./python.exe`) or `PCbuild\build.bat -c Debug` (Windows, then `python.bat`). Debug builds add checks for development. Source becomes tokens (`tokenize`), a syntax tree (`ast`), a code object (`compile`, `__code__`) and runs in `ceval.c`; never test exact bytecode. Run `./python -m test`, read its summary, and compare failures with a baseline on unchanged `main`.

### Exercise: The build commands

Write `build_commands(system, debug=True, jobs=8)` returning a list of commands. For `"linux"` and `"macos"`: `./configure --with-pydebug` (just `./configure` when not debug), `make -jJOBS`, and `BINARY -m test -jJOBS`, where BINARY is `./python` on Linux and `./python.exe` on macOS. For `"windows"`: `PCbuild\build.bat -c Debug` (or `-c Release`) and `python.bat -m test -jJOBS`. Raise `ValueError` for any other system.

In [ ]:
def build_commands(system, debug=True, jobs=8):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_commands():\n    \"per system\"\n    assert build_commands(\"linux\") == [\"./configure --with-pydebug\", \"make -j8\", \"./python -m test -j8\"]\n    assert build_commands(\"macos\", jobs=4) == [\"./configure --with-pydebug\", \"make -j4\", \"./python.exe -m test -j4\"]\n    assert build_commands(\"linux\", debug=False)[0] == \"./configure\"\n    assert build_commands(\"windows\") == [\"PCbuild\\\\build.bat -c Debug\", \"python.bat -m test -j8\"]\n    assert build_commands(\"windows\", debug=False, jobs=2) == [\"PCbuild\\\\build.bat -c Release\", \"python.bat -m test -j2\"]\n    try:\n        build_commands(\"amiga\")\n    except ValueError:\n        pass\n    else:\n        raise AssertionError(\"amiga\")")

### Exercise: Which interpreter is this?

Write `parse_sys_version(text)` for strings like `3.15.0a1+ (heads/main:1a2b3c4, Nov 20 2026, 14:03:12) [GCC 13.2.0]`: return a dict with `version` (before the space), `branch` (between `heads/` and `:`, or `None` if there's no `heads/`), `commit` (after the colon, or `None`), `built` (the date and time, as written) and `compiler` (inside the square brackets).

In [ ]:
import re

def parse_sys_version(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_versions():\n    \"your build, or a release\"\n    assert parse_sys_version(\"3.15.0a1+ (heads/main:1a2b3c4, Nov 20 2026, 14:03:12) [GCC 13.2.0]\") == {\n        \"version\": \"3.15.0a1+\", \"branch\": \"main\", \"commit\": \"1a2b3c4\", \"built\": \"Nov 20 2026, 14:03:12\", \"compiler\": \"GCC 13.2.0\"}\n    assert parse_sys_version(\"3.14.1 (heads/fix-shorten:9f8e7d6, Dec  1 2026, 09:15:00) [Clang 17.0.6 ]\")[\"branch\"] == \"fix-shorten\"\n    assert parse_sys_version(\"3.13.2 (main, Feb  4 2026, 12:00:00) [GCC 12.2.0]\") == {\n        \"version\": \"3.13.2\", \"branch\": None, \"commit\": None, \"built\": \"Feb  4 2026, 12:00:00\", \"compiler\": \"GCC 12.2.0\"}")

### Exercise: Tokens

Write `token_kinds(source)` with `tokenize`: return `(KIND, STRING)` pairs for every token whose kind is `NAME`, `OP`, `NUMBER`, `STRING`, `INDENT` or `DEDENT` (using `tokenize.tok_name` for the kind), in order. For `INDENT` and `DEDENT`, use an empty string.

In [ ]:
import io
import tokenize

def token_kinds(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_tokens():\n    \"the first stage\"\n    assert token_kinds(\"total = km * 12\\n\") == [(\"NAME\", \"total\"), (\"OP\", \"=\"), (\"NAME\", \"km\"), (\"OP\", \"*\"), (\"NUMBER\", \"12\")]\n    assert token_kinds(\"if x:\\n    y('ok')\\n\") == [(\"NAME\", \"if\"), (\"NAME\", \"x\"), (\"OP\", \":\"), (\"INDENT\", \"\"), (\"NAME\", \"y\"), (\"OP\", \"(\"),\n                                               (\"STRING\", \"'ok'\"), (\"OP\", \")\"), (\"DEDENT\", \"\")]")

### Exercise: Inside a code object

Write `code_summary(func)`: from `func.__code__`, return a dict with `arguments` (the names of its positional arguments, from `co_varnames` and `co_argcount`, as a list), `locals` (its other local variable names, as a list), and `globals` (the names it looks up that aren't local, from `co_names`, as a sorted list). Then `module_names(source)`: the sorted names in `co_names` of `compile(source, "<module>", "exec")`.

In [ ]:
def code_summary(func):
    ___

def module_names(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import math\n\ndef walk_minutes(km, climb_m):\n    flat = km / 5 * 60\n    hills = climb_m / 10\n    return round(flat + hills)\n\ndef test_summary():\n    \"what the compiler recorded\"\n    assert code_summary(walk_minutes) == {\"arguments\": [\"km\", \"climb_m\"], \"locals\": [\"flat\", \"hills\"], \"globals\": [\"round\"]}\n    def hypot(a, b):\n        return math.sqrt(a * a + b * b)\n    assert code_summary(hypot) == {\"arguments\": [\"a\", \"b\"], \"locals\": [], \"globals\": [\"math\", \"sqrt\"]}\n\ndef test_module():\n    \"a module's names\"\n    assert module_names(\"import math\\nx = math.sqrt(y)\\n\") == [\"math\", \"sqrt\", \"x\", \"y\"]")

### Exercise: Reading the test summary

Write `parse_summary(output)` for summaries like those in `runs.py`: return a dict with `result` (from `== Tests result: X ==`), `ok` (the number from `N tests OK.` or `1 test OK.`, else 0), `failed` and `skipped` (lists of the test names listed under `N tests failed:` and `N tests skipped:`, which may run over several indented lines; empty if absent), and `duration` (after `Total duration: `, or `None`).

In [ ]:
import re

def parse_summary(output):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from runs import BASELINE, AFTER_CHANGE\n\ndef test_baseline():\n    \"unchanged main\"\n    assert parse_summary(BASELINE) == {\"result\": \"FAILURE\", \"ok\": 433, \"failed\": [\"test_ssl\"],\n                                       \"skipped\": [\"test_devpoll\", \"test_kqueue\", \"test_msvcrt\", \"test_winreg\", \"test_winsound\"],\n                                       \"duration\": \"4 min 12 sec\"}\n\ndef test_after():\n    \"with a change\"\n    s = parse_summary(AFTER_CHANGE)\n    assert s[\"failed\"] == [\"test_ssl\", \"test_textwrap\", \"test_pydoc\"] and s[\"ok\"] == 431 and len(s[\"skipped\"]) == 4\n\ndef test_success():\n    \"all passing\"\n    assert parse_summary(\"== Tests result: SUCCESS ==\\n\\n1 test OK.\\n\") == {\"result\": \"SUCCESS\", \"ok\": 1, \"failed\": [], \"skipped\": [], \"duration\": None}")

### Bug hunt: No ./python on a Mac

On macOS the built interpreter is `./python.exe`, but this gives everyone `./python`, so it prints `./python -m test`. Fix it so it prints `./python.exe -m test` for macOS (and still `./python` on Linux).

In [ ]:
def test_command(system):
    return "./python -m test"

print(test_command("macos"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"macOS's binary\"\n    expect_output(\"./python.exe -m test\")\n    assert test_command(\"linux\") == \"./python -m test\"")

### Bug hunt: Two failures that looked like one

The failed tests are separated by spaces, not commas, so splitting on `", "` keeps them together and this prints `1`. Split on whitespace so it prints `2`.

In [ ]:
line = "    test_ssl test_textwrap"
print(len(line.strip().split(", ")))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"split()\"\n    expect_output(\"2\")")

### Puzzle: Commands for Linux and macOS

Arrange `build_commands` for Linux and macOS: configure (with the debug flag if asked), make, then run the tests with the right binary. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the configure option for a debug build
q = ___   # the name of the built interpreter on macOS, as you'd run it
r = ___   # the C file whose loop runs bytecode, as a path
s = ___   # True or False: a test may safely check exact bytecode instructions
t = ___   # walk_minutes.__code__.co_argcount for def walk_minutes(km, climb_m)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"--with-pydebug\" and q == \"./python.exe\" and r == \"Python/ceval.c\", \"p, q and r\"\n    assert s is False and t == 2, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: was it my change?

Write `compare_runs(baseline_output, change_output)` using `parse_summary` (given). Return `(lines, verdict)`:

- `"new failures: ..."`: tests failing with the change but not on the baseline (in the change run's order);
- `"failing on main too: ..."`: tests failing in both;
- `"fixed: ..."`: tests failing on the baseline but not with the change;
- `"skips changed: ..."`: tests skipped in one run but not the other, sorted, each followed by ` (now skipped)` or ` (no longer skipped)`;
- `"passing: N (was M)"` from the `ok` counts.

Write `none` for an empty list. `verdict` is `"your change looks safe"` if there are no new failures, otherwise `"investigate: "` and the new failures joined by `", "`.

In [ ]:
import re

def parse_summary(output):
    summary = {"result": None, "ok": 0, "failed": [], "skipped": [], "duration": None}
    current = None
    for line in output.splitlines():
        if match := re.match(r"== Tests result: (\w+) ==", line):
            summary["result"], current = match.group(1), None
        elif match := re.match(r"(\d+) tests? OK\.", line):
            summary["ok"], current = int(match.group(1)), None
        elif match := re.match(r"\d+ tests? (failed|skipped):", line):
            current = match.group(1)
        elif line.startswith(" ") and current and line.strip():
            summary[current] += line.split()
        elif line.startswith("Total duration: "):
            summary["duration"], current = line[len("Total duration: "):].strip(), None
        elif not line.strip():
            current = None
    return summary

def compare_runs(baseline_output, change_output):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from runs import BASELINE, AFTER_CHANGE\n\ndef test_compare():\n    \"two failures to investigate\"\n    lines, verdict = compare_runs(BASELINE, AFTER_CHANGE)\n    assert lines == [\"new failures: test_textwrap, test_pydoc\", \"failing on main too: test_ssl\", \"fixed: none\",\n                     \"skips changed: test_winsound (no longer skipped)\", \"passing: 431 (was 433)\"]\n    assert verdict == \"investigate: test_textwrap, test_pydoc\"\n\ndef test_safe():\n    \"nothing new broke\"\n    lines, verdict = compare_runs(AFTER_CHANGE, BASELINE)\n    assert lines[:3] == [\"new failures: none\", \"failing on main too: test_ssl\", \"fixed: test_textwrap, test_pydoc\"]\n    assert lines[3] == \"skips changed: test_winsound (now skipped)\" and verdict == \"your change looks safe\"")

## Warm-up

`sys.version` describes the interpreter you're running: its version, where it was built from and when, and the compiler. When you build CPython yourself, it names your branch and commit, which is how you know you're testing your own build.

## Building it

The devguide's "Setup and building" page has the details and the list of libraries to install first. In outline, from the top of your clone:

- **Linux**: `./configure --with-pydebug` then `make -j8` (use as many jobs as you have cores). The new interpreter is `./python`.
- **macOS**: the same, but the interpreter is `./python.exe` (macOS file systems usually ignore case, so `python` would clash with the `Python` folder).
- **Windows**: `PCbuild\build.bat -c Debug`, then run it with `python.bat`.

`--with-pydebug` makes a **debug build**: slower, but with extra checks that catch mistakes in C code early, and extra tools such as `sys.gettotalrefcount()`. It's what you want while developing. Rebuilding after a change to a `.py` file in `Lib/` needs nothing at all; after a change to C code, run `make` again, which only rebuilds what changed.

## From source to running code

What happens when the interpreter runs your file is a pipeline you can watch from Python:

1. **Tokens**: the `tokenize` module splits source into names, operators, numbers and strings, plus `INDENT` and `DEDENT` for the blocks.
2. **Syntax tree**: the parser (generated from `Grammar/python.gram`) builds the tree that `ast.parse` shows you.
3. **Code object**: the compiler turns the tree into bytecode with tables of names and constants: `compile(source, "file", "exec")` returns it, and a function's is in `func.__code__`.
4. **Evaluation**: the loop in `Python/ceval.c` runs the bytecode. The `dis` module shows it, though the exact instructions change from version to version, so never write a test that depends on them.

Knowing the stages tells you where to look: a new syntax error message lives in the parser, a wrong result from `sorted` lives in `Objects/listobject.c`, a bug in `textwrap` lives in `Lib/textwrap.py`.

## Running the tests

`./python -m test` runs the whole suite (add `-j0` to use every core); `./python -m test test_textwrap` runs one module, and `-v` shows each test. The summary at the end says how many test modules passed, which **failed** and which were **skipped** (a test for Windows registry access is skipped on Linux). It looks something like the runs in `runs.py`, though the exact wording varies between versions.

Before blaming your change for a failure, run the same tests on an unchanged `main` (a **baseline**). A test that fails there too isn't yours (perhaps your computer lacks a network or a library), though it's worth reporting if nobody has. A test that fails only with your change is yours to fix.

## Practice

## Quiz

## Daily challenge

## Your turn: build it

1. **Install the build dependencies** for your system from the devguide's setup page.
2. **Build a debug CPython** from your clone, and check `./python -c "import sys; print(sys.version)"` names your branch.
3. **Run the whole suite** with `-j0` and save its summary as your baseline. Note anything that fails or is skipped, and why.
4. **Make a harmless change** (add a comment to `Lib/textwrap.py`), run `test_textwrap` again, and compare.

## Before you go

Tomorrow you'll find an issue to work on: one that's small, wanted, and not already taken.